# KLIFS structure filtering by UniProt

This notebook pulls kinase structures from the KLIFS API and keeps only structures that satisfy:
- `resolution <= 2.5`
- `missing_residues <= 5`
- `quality_score >= 8.0` (KLIFS field corresponding to quality)
- valid `uniprot` from `kinase_information`

If multiple structures map to the same UniProt accession, the best structure is selected by:
1. highest quality score
2. lowest resolution
3. least missing residues

In [1]:
import requests
import pandas as pd
from typing import Iterable, List

BASE_URL = "https://klifs.net/api"
OUTPUT_CSV = "klifs_filtered_structures_uniprot.csv"
REQUEST_TIMEOUT = 60
KINASE_CHUNK_SIZE = 75  # avoid very long query URLs

session = requests.Session()
session.headers.update({"Accept": "application/json"})

In [2]:
def chunked(values: List[int], size: int) -> Iterable[List[int]]:
    for i in range(0, len(values), size):
        yield values[i:i + size]


def get_json(endpoint: str, params: dict | None = None):
    response = session.get(f"{BASE_URL}{endpoint}", params=params, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()


def fetch_human_kinase_information() -> pd.DataFrame:
    kinase_info = get_json("/kinase_information", params={"species": "HUMAN"})
    kinase_df = pd.DataFrame(kinase_info)

    required = {"kinase_ID", "uniprot"}
    missing = required - set(kinase_df.columns)
    if missing:
        raise ValueError(f"Missing required columns in kinase_information response: {missing}")

    kinase_df = kinase_df[["kinase_ID", "name", "species", "uniprot"]].copy()
    kinase_df["kinase_ID"] = pd.to_numeric(kinase_df["kinase_ID"], errors="coerce").astype("Int64")
    return kinase_df.dropna(subset=["kinase_ID"])


def fetch_structures_for_kinases(kinase_ids: List[int]) -> pd.DataFrame:
    all_structures = []

    for ids in chunked(kinase_ids, KINASE_CHUNK_SIZE):
        params = {"kinase_ID": ",".join(map(str, ids))}
        batch = get_json("/structures_list", params=params)
        if isinstance(batch, list):
            all_structures.extend(batch)

    return pd.DataFrame(all_structures)

In [3]:
# 1) Pull kinase info (contains UniProt)
kinase_df = fetch_human_kinase_information()

# 2) Pull all structures for those kinase IDs
kinase_ids = kinase_df["kinase_ID"].dropna().astype(int).unique().tolist()
structures_df = fetch_structures_for_kinases(kinase_ids)

# 3) Keep only fields needed for filtering/ranking/output
wanted_cols = [
    "structure_ID",
    "kinase_ID",
    "kinase",
    "species",
    "pdb",
    "chain",
    "resolution",
    "quality_score",
    "missing_residues",
]
present_cols = [c for c in wanted_cols if c in structures_df.columns]
structures_df = structures_df[present_cols].copy()

# 4) Merge UniProt onto structures
merged = structures_df.merge(
    kinase_df[["kinase_ID", "uniprot"]],
    on="kinase_ID",
    how="left"
)

# 5) Normalize numeric columns
for col in ["resolution", "quality_score", "missing_residues"]:
    if col in merged.columns:
        merged[col] = pd.to_numeric(merged[col], errors="coerce")

# 6) Apply hard filters
filtered = merged.loc[
    (merged["resolution"] <= 2.5)
    & (merged["missing_residues"] <= 5)
    & (merged["quality_score"] >= 8.0)
    & (merged["uniprot"].notna())
    & (merged["uniprot"].astype(str).str.strip() != "")
].copy()

# 7) Resolve non-unique UniProt: best quality, then best resolution, then least missing residues
best_per_uniprot = (
    filtered
    .sort_values(
        by=["uniprot", "quality_score", "resolution", "missing_residues"],
        ascending=[True, False, True, True],
        kind="mergesort"
    )
    .drop_duplicates(subset=["uniprot"], keep="first")
    .sort_values("uniprot")
    .reset_index(drop=True)
)

best_per_uniprot.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(best_per_uniprot):,} structures to {OUTPUT_CSV}")
best_per_uniprot.head(10)

Saved 225 structures to klifs_filtered_structures_uniprot.csv


,structure_ID,kinase_ID,kinase,species,pdb,chain,resolution,quality_score,missing_residues,uniprot
0,1500,520,BMPR1B,Human,3mdy,C,2.05,9.6,0,O00238
1,13841,267,CDC7,Human,6ya6,A,1.44,8.0,0,O00311
2,1818,314,PLK4,Human,3cok,A,2.25,8.8,0,O00444
3,463,381,YSK1,Human,2xik,A,1.97,9.2,0,O00506
4,12695,389,MAP2K7,Human,6yg0,A,2.00,9.6,0,O14733
5,13496,121,CHK1,Human,7ako,B,1.80,9.9,0,O14757
6,13583,139,CASK,Human,7oai,A,2.30,9.7,0,O14936
7,1977,259,AurA,Human,1mq4,A,1.90,9.6,0,O14965
8,336,279,GAK,Human,4c58,A,2.16,9.4,0,O14976
9,6987,145,DCLK1,Human,5jzj,A,1.71,9.4,0,O15075
